# INTERVENE — write residual or logits

Same code as `scripts/intervene.py`, one **mode per cell**. Qwen weights stay frozen.

**Kernel:** `CXR local Qwen (faiss_gpu1)`

Run order:
1. **Setup**
2. **Load model** (skip if you already loaded it in another notebook in this kernel — this kernel is per notebook, so load once here)
3. Any mode cell below


## Setup


In [ ]:
import json
import os
import sys
from pathlib import Path

SCRIPTS = Path("/home/udonsi-kalu/staging/cxr-mi-repeng-grounding/scripts")
CASES = Path("/home/udonsi-kalu/staging/cxr-mi-repeng-grounding/learning_lab/cases/oncology_m1.json")

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import look
import intervene
import process
from _common import PROMPT_A, PROMPT_B, PROMPT_TEST, DEFAULT_MODEL, boot, encode, last_residual, device

cases = json.loads(CASES.read_text(encoding="utf-8"))
NOTE = cases["foundation_note"]
LAYER = 20
MAX_NEW = 24

print("Teaching note:", NOTE)
print("Imported look, intervene, process.")


## Load model (run once)


In [ ]:
model, tok = boot(DEFAULT_MODEL, layer=LAYER)


### `steer`


In [ ]:
intervene.cmd_steer(model, tok, layer=LAYER, alpha=8.0, max_new=MAX_NEW)


### `patch`


In [ ]:
intervene.cmd_patch(model, tok, layer=LAYER, max_new=MAX_NEW)


### `mlp_zero`


In [ ]:
intervene.cmd_zero(model, tok, layer=LAYER, max_new=MAX_NEW, site="mlp")


### `attn_zero`


In [ ]:
intervene.cmd_zero(model, tok, layer=LAYER, max_new=MAX_NEW, site="attn")


### `resid_zero`


In [ ]:
intervene.cmd_zero(model, tok, layer=LAYER, max_new=MAX_NEW, site="block")


### `logit_bias`


In [ ]:
intervene.cmd_logit_bias(model, tok, max_new=MAX_NEW)
